Approach:

Chapter → Chapter Recommendation:
We first transform each chapter’s tags into a clean textual format (tags_sent) and convert them into numerical vectors using TF-IDF, which captures the importance of each tag across all chapters. Using these vectors, we train a Nearest Neighbors model with cosine distance to measure similarity between chapters in the same feature space. For every chapter, we retrieve its nearest neighbors, filter out the chapter itself and any duplicates, and keep the top-k most similar chapters. Finally, we convert distances into similarity scores and attach these recommendations back to df_chapter, giving us, for each chapter, a list of similar chapters along with how similar they are. This effectively enables content-based recommendations at the chapter level based on shared themes.

Chapter → Book Recommendation (Query-based):
We then shift from chapter-level similarity to cross-level retrieval by constructing book-level representations. We aggregate all tags from chapters belonging to the same book, clean and deduplicate them, and create a single text representation (book_tags) per book. To ensure both chapters and books lie in the same feature space, we fit a TF-IDF vectorizer on the combined corpus of chapter and book tags, and generate embeddings for both. We train a Nearest Neighbors model on book vectors and query it using chapter vectors, which allows us to find the most relevant books for each chapter based on thematic similarity. These recommended books and their distances are then mapped back to df_chapter and further propagated to df_interaction, enabling personalized book recommendations grounded in the content of chapters users have interacted with.




#### Import Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### File Path to the chapter.csv and interactions.csv file

Instruction to run : Upload the notebook to Google Colab and upload the chapters.csv and interactions.csv files to your Google Drive. Then, update the file paths in the notebook to point to the correct locations of these files in your Google Drive. Then run the notebook.

In [2]:
chapters = "/content/drive/MyDrive/Pratilipi/chapters.csv"
interactions = "/content/drive/MyDrive/Pratilipi/interactions.csv"

### read both the CSV file using pandas

In [3]:
import pandas as pd

df_chapter = pd.read_csv(chapters)
df_interaction = pd.read_csv(interactions)

In [4]:
#df_chp = pd.DataFrame(df_chapter.loc[:, ['book_id', "chapter_sequence_no", "chapter_id"]].groupby(["book_id", "chapter_sequence_no", "chapter_id"]).count().reset_index())

### Create a new columns 'tags_sent' such that it replaces white space of tags column with underscore and pipe symbol with white spaces

In [7]:
df_chapter['tags1'] = df_chapter['tags'].apply(
    lambda x: [tag.replace(" ", "_") for tag in x.split("|")]
)

In [8]:
df_chapter["tags_sent"] = df_chapter["tags1"].apply(lambda x: " ".join(x))

### We build a content-based recommendation system that finds similar chapters based on their tags.

### We convert chapter tags into vector representations using TF-IDF, use a nearest neighbor model to find the most similar chapters based on cosine similarity, filter out the chapter itself and duplicates, and store the top-k similar chapters along with their similarity scores in the dataset.

### Example:

Chapter A → "Fantasy Horror"
Chapter B → "Fantasy"
Chapter C → "Horror Thriller"

👉 After processing:

Chapter A → [Chapter B, Chapter C]
Similarity → [0.85, 0.78]

👉 Meaning:

We identified that Chapter B and Chapter C are most similar to Chapter A based on their tags.

### Foe evaluation of chapter to chapter distance, we use cosine similarity. This is how we measure quality of recommendation

In [9]:
# ####
# Step 1 : We convert the textual tags of each chapter into numerical vectors using TF-IDF.
# We represent each chapter as a vector in a high-dimensional space where each dimension corresponds to a tag.

# Step 2: We initialize a Nearest Neighbors model to compute similarity between chapters.
# We use cosine distance because it works well for text-based vector representations.

# Step3 : We fit the model on the TF-IDF matrix so it can learn the vector space of chapters.

# Step 4 : We compute the nearest neighbors for each chapter.
# For every chapter, we get:
#   - indices → positions of similar chapters
#   - distances → how far they are in vector space

# Step 5 : We extract the actual chapter IDs to map model indices to real-world identifiers.

# Step 6: We iterate over each chapter to build its list of similar chapters.

# Step 7 : We initialize containers to store unique similar chapters and their distances.

# Step 8 : We iterate over the nearest neighbors of the current chapter.

# Step 9 : We map each neighbor’s row index to its corresponding chapter ID.

# Step 10 : We remove the chapter itself from its own recommendations.

# Step 11 : We avoid duplicate chapter recommendations.

# Step 12: We store valid similar chapters and their distances.

# Step 13 : We stop once we have collected the top_k recommendations.

# Step 14 : We store the results for the current chapter.

# Step 15 : We convert cosine distance into similarity scores for evaluation metric

# Step 16 : We enrich the original DataFrame by adding similar chapters, distances, and similarity scores for each chapter.


import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# -----------------------------
# 1. TF-IDF Vectorization
# -----------------------------
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df_chapter["tags_sent"])

# -----------------------------
# 2. Nearest Neighbors Model
# -----------------------------
top_k = 5
candidate_k = top_k   # extra neighbors to handle duplicates

nn = NearestNeighbors(
    n_neighbors=candidate_k,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

nn.fit(tfidf_matrix)

# -----------------------------
# 3. Get Neighbors
# -----------------------------
distances, indices = nn.kneighbors(tfidf_matrix)

# -----------------------------
# 4. Map to chapter_id + filter self
# -----------------------------
chapter_ids = df_chapter["chapter_id"].values

final_chapters = []
final_distances = []

for i in range(len(indices)):
    seen = set()
    row_chapters = []
    row_dist = []

    for idx, dist in zip(indices[i], distances[i]):
        ch_id = chapter_ids[idx]


        if idx == i:
            continue


        if ch_id in seen:
            continue

        seen.add(ch_id)
        row_chapters.append(ch_id)
        row_dist.append(dist)

        if len(row_chapters) == top_k:
            break

    final_chapters.append(row_chapters)
    final_distances.append(row_dist)

# -----------------------------
# 5. Convert to similarity
# -----------------------------
final_similarity = [[1 - d for d in row] for row in final_distances]

# -----------------------------
# 6. Attach to DataFrame
# -----------------------------
df_chapter["similar_chapters"] = final_chapters
df_chapter["similar_distances"] = final_distances
df_chapter["similarity_scores"] = final_similarity

### Test case/Results: For `chapter_id` = 2812946, we show the list of similar chapters i.e. [ 8537289, 3161921, 3980779, 1146761, 3613347]. The `chapter_id` = 2812946 has `tags_sent` = "Fantasy Horror". Similar chapters [ 8537289, 3161921, 3980779, 1146761, 3613347] also have `tag_sent` = "Fantasy Horror"

---



In [10]:
df_chapter.loc[df_chapter['chapter_id'].isin([2812946, 8537289, 3161921, 3980779, 1146761, 3613347]), :]

,chapter_id,chapter_sequence_no,book_id,author_id,published_date,tags,tags1,tags_sent,similar_chapters,similar_distances,similarity_scores
0,2812946,1,139726,66847,1990-03-22,Fantasy|Horror,"[Fantasy, Horror]",Fantasy Horror,"[8537289, 3161921, 3980779, 1146761, 3613347]","[0.0, 0.0, 0.0, 0.0, 0.0]","[1.0, 1.0, 1.0, 1.0, 1.0]"
7911,3161921,1,484914,49221,2003-09-27,Horror|Fantasy,"[Horror, Fantasy]",Horror Fantasy,"[8537289, 3980779, 1146761, 3613347]","[0.0, 0.0, 0.0, 0.0]","[1.0, 1.0, 1.0, 1.0]"
11002,1146761,5,269172,92203,2001-05-09,Fantasy|Horror,"[Fantasy, Horror]",Fantasy Horror,"[8537289, 3161921, 3980779, 3613347]","[0.0, 0.0, 0.0, 0.0]","[1.0, 1.0, 1.0, 1.0]"
21947,3613347,3,392438,18981,1998-04-02,Horror|Fantasy,"[Horror, Fantasy]",Horror Fantasy,"[8537289, 3161921, 3980779, 1146761]","[0.0, 0.0, 0.0, 0.0]","[1.0, 1.0, 1.0, 1.0]"
40701,3980779,2,836873,12128,1994-08-25,Horror|Fantasy,"[Horror, Fantasy]",Horror Fantasy,"[8537289, 3161921, 1146761, 3613347]","[0.0, 0.0, 0.0, 0.0]","[1.0, 1.0, 1.0, 1.0]"
45935,8537289,1,244935,16839,2009-03-28,Fantasy|Horror,"[Fantasy, Horror]",Fantasy Horror,"[3161921, 3980779, 1146761, 3613347]","[0.0, 0.0, 0.0, 0.0]","[1.0, 1.0, 1.0, 1.0]"


### We transform chapter-level tags into clean, deduplicated book-level tag representations. We split and clean chapter-level tags, aggregate them at the book level, remove duplicates, and convert them into a standardized text format suitable for downstream modeling.

Example :

Input (chapters tag):
"Literary Fiction|Graphic Novel"
"Dystopian|Literary Fiction"

Output (book level tag):
"Literary_Fiction Graphic_Novel Dystopian"

In [11]:
df_book_tags = (
    df_chapter
    .assign(
        # 1. split tags → list
        tags_split=lambda df: df["tags"].str.split("|")
    )
    .explode("tags_split")  # 2. one tag per row
    .assign(
        # 3. clean tag (replace spaces with underscore)
        tags_clean=lambda df: df["tags_split"]
            .str.strip()
            .str.replace(" ", "_")
    )
    .groupby("book_id")["tags_clean"]
    .apply(lambda x: list(set(x)))  # 4. remove duplicates
    .apply(lambda x: " ".join(x))   # 5. join with space
    .reset_index(name="book_tags")
)

### We fit a single TF-IDF vectorizer on the combined corpus of chapter and book tags to create a shared feature space, and then transform chapters and books into comparable vector representations.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

# Fit on combined corpus (VERY IMPORTANT)
tfidf.fit(
    pd.concat([df_chapter["tags_sent"], df_book_tags["book_tags"]])
)

chapter_vec = tfidf.transform(df_chapter["tags_sent"])
book_vec = tfidf.transform(df_book_tags["book_tags"])

### We initialize and fit a Nearest Neighbors model on the book vectors using cosine distance to enable retrieval of the top-k most similar books.

In [13]:
from sklearn.neighbors import NearestNeighbors

top_k = 5

nn = NearestNeighbors(
    n_neighbors=top_k,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

nn.fit(book_vec)  # IMPORTANT: fit on BOOKS

NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1)

### We query the trained nearest neighbors model with chapter vectors to retrieve the top-k most similar books along with their distances.

Assumption : Since, user has not read entire book(for most of the cases), chapter vector is more appropriate for querying book using cosine similarity.

In [14]:
distances, indices = nn.kneighbors(chapter_vec)

### We map nearest neighbor indices to their corresponding book_ids and simultaneously collect the associated distances for each chapter.

In [15]:
book_ids = df_book_tags["book_id"].values

chapter_to_books = []
chapter_to_distances = []
for i in range(len(indices)):
    row_books = [book_ids[idx] for idx in indices[i]]
    chapter_to_books.append(row_books)
    row_dist = [dist for dist in distances[i]]
    chapter_to_distances.append(row_dist)

### We attach recommended books and their distances to each chapter, create mappings from chapter_id to these recommendations, and propagate them to the interaction dataset based on chapter_id.

### For evaluation we use cosine similarity.

In [16]:
df_chapter["recommended_books"] = chapter_to_books
df_chapter["chapter_to_book_distances"] = chapter_to_distances

In [17]:
chapter_to_books_map = dict(
    zip(df_chapter["chapter_id"], df_chapter["recommended_books"])
)

# create mapping for distances
chapter_to_dist_map = dict(
    zip(df_chapter["chapter_id"], chapter_to_distances)
)

df_interaction["recommended_books"] = df_interaction["chapter_id"].map(chapter_to_books_map)
df_interaction["chapter_to_book_distances"] = df_interaction["chapter_id"].map(chapter_to_dist_map)

In [18]:
df_interaction

,user_id,chapter_id,book_id,recommended_books,chapter_to_book_distances
0,user_2378720,5894067,444295,"[736356, 834335, 252203, 863998, 597012]","[0.0, 0.0, 0.0, 0.0, 0.0]"
1,user_2321122,2532511,785684,"[609442, 723861, 312861, 522700, 346729]","[0.0, 0.0, 0.0, 0.0, 0.0]"
2,user_2335775,6777764,999595,"[139186, 927175, 359168, 298775, 775990]","[0.0, 0.1311815567030532, 0.13205153291456617,..."
3,user_7906001,7366896,748410,"[783185, 532876, 977057, 600318, 169465]","[0.0, 0.1331250611883641, 0.13391215105022836,..."
4,user_9981689,7853186,418083,"[551883, 135253, 970645, 728765, 734088]","[0.0, 0.0, 0.0, 0.0, 0.18141873864252178]"
...,...,...,...,...,...
999995,user_5216801,7726893,340860,"[100089, 780081, 889327, 122238, 667430]","[0.0, 0.0, 0.0, 0.18012794513749708, 0.1810175..."
999996,user_8074162,3705751,160045,"[762177, 560438, 354016, 520106, 250703]","[0.0, 0.0, 0.0, 0.0, 0.0]"
999997,user_8348304,1044919,912767,"[195189, 826515, 847986, 554691, 830549]","[0.0, 0.10399433647735412, 0.10413037977166884..."
999998,user_6924764,6254645,648779,"[998630, 101657, 675507, 300444, 814562]","[0.0, 0.0, 0.0, 0.0, 0.0]"


### Merge chapter and book_tags dataframe into one


In [19]:
df_final = pd.merge(df_chapter, df_book_tags, on = ["book_id"], how = "inner")

In [20]:
df_final_1= df_final.loc[:, [ "book_id", "chapter_id", "tags_sent", "recommended_books", "book_tags"]]

In [21]:
df_final_1

,book_id,chapter_id,tags_sent,recommended_books,book_tags
0,139726,2812946,Fantasy Horror,"[893853, 722571, 736667, 849149, 650472]",Literary_Fiction Young_Adult Fantasy Horror
1,139726,4330764,Fantasy Young_Adult Literary_Fiction,"[799806, 457833, 894617, 550956, 944510]",Literary_Fiction Young_Adult Fantasy Horror
2,139726,2664499,Fantasy,"[305159, 989322, 492850, 490394, 509451]",Literary_Fiction Young_Adult Fantasy Horror
3,139726,2260666,Literary_Fiction Fantasy,"[521530, 417358, 507000, 390206, 536191]",Literary_Fiction Young_Adult Fantasy Horror
4,191772,6069976,Horror Young_Adult Romance Graphic_Novel,"[627054, 848226, 913274, 659461, 286710]",Horror Paranormal Graphic_Novel Literary_Ficti...
...,...,...,...,...,...
49995,493482,6077160,Historical_Fiction Humor Fantasy Paranormal,"[176115, 985194, 166309, 533672, 804179]",Historical_Fiction Humor Horror Thriller Dysto...
49996,493482,5508465,Young_Adult Historical_Fiction,"[580419, 223949, 881565, 169429, 866296]",Historical_Fiction Humor Horror Thriller Dysto...
49997,493482,7383582,Dystopian Historical_Fiction,"[102284, 308252, 940494, 539963, 675512]",Historical_Fiction Humor Horror Thriller Dysto...
49998,428817,4413776,Humor,"[998630, 101657, 675507, 300444, 814562]",Humor


### Test case: for Book id 191772 and chapter id 6069976 show the similar books. The chapter_id 6069976 and book_id 191772 have "Horror Young_Adult Romance Graphic_Novel" tags. Similar Books had the below tags:
- Humor Horror Graphic_Novel Romance Young_Adult
- Horror Graphic_Novel Romance
- Horror Literary_Fiction Graphic_Novel Romance
- Graphic_Novel Romance Young_Adult Horror
- Graphic_Novel Romance Horror

In [22]:
df_final_1.loc[(df_final_1['book_id'] == 191772) & (df_final_1['chapter_id'] == 6069976), :]

,book_id,chapter_id,tags_sent,recommended_books,book_tags
4,191772,6069976,Horror Young_Adult Romance Graphic_Novel,"[627054, 848226, 913274, 659461, 286710]",Horror Paranormal Graphic_Novel Literary_Ficti...


In [23]:
df_final_1.loc[(df_final_1['book_id'].isin([627054, 848226, 913274, 659461, 286710])), :]

,book_id,chapter_id,tags_sent,recommended_books,book_tags
3504,913274,6585048,Horror Humor Graphic_Novel,"[348918, 975744, 439121, 125918, 506045]",Humor Horror Graphic_Novel Romance Young_Adult
3505,913274,5017375,Graphic_Novel Horror Romance Young_Adult,"[627054, 848226, 913274, 659461, 286710]",Humor Horror Graphic_Novel Romance Young_Adult
3506,913274,3686938,Horror,"[832162, 982725, 492906, 894348, 276470]",Humor Horror Graphic_Novel Romance Young_Adult
17545,286710,6240092,Romance Horror,"[165214, 993719, 764001, 876383, 700896]",Horror Graphic_Novel Romance
17546,286710,1573502,Horror Graphic_Novel,"[994141, 525581, 261177, 186518, 365302]",Horror Graphic_Novel Romance
24581,848226,7911611,Romance,"[741199, 610626, 935527, 143892, 292818]",Horror Literary_Fiction Graphic_Novel Romance ...
24582,848226,2824332,Romance,"[741199, 610626, 935527, 143892, 292818]",Horror Literary_Fiction Graphic_Novel Romance ...
24583,848226,9888216,Romance Horror,"[165214, 993719, 764001, 876383, 700896]",Horror Literary_Fiction Graphic_Novel Romance ...
24584,848226,2359591,Romance,"[741199, 610626, 935527, 143892, 292818]",Horror Literary_Fiction Graphic_Novel Romance ...
24585,848226,6011826,Romance Young_Adult Graphic_Novel Literary_Fic...,"[493914, 691123, 848226, 311118, 301814]",Horror Literary_Fiction Graphic_Novel Romance ...
